# PDF reporting for HDAB id project

This script uses Python and the ReportLab toolkit https://www.reportlab.com/, plus some basic modules. 


To do:
- Link to database when available based on view with left join morphospeciescodes
- Add verbosity
- Create functions that can run as a script with argparse
- Loop functions over morphospecies in pulled dataframe to create PDF for each row; order by HDOAref
- Add argument to combine multiple PDFs into one: https://dadataguy.medium.com/merging-multiple-pdfs-with-python-7970a720ff0f
- Add a wishlist option to supply a csv with list of morphospeciescodes to generate report for
- Add argument to set directory with images

## Import dependencies, fonts, and set working directories

In [387]:
import os
import pandas as pd
import reportlab
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from datetime import date
from pathlib import Path

font_dir = "./fonts/" 
arial_reg = os.path.join(font_dir, "Arial.ttf")
arial_bold = os.path.join(font_dir, "Arial Bold.ttf")
arial_italic = os.path.join(font_dir, "Arial Italic.ttf")
arial_bolditalic = os.path.join(font_dir, "Arial Bold Italic.ttf")

pdfmetrics.registerFont(TTFont('Arial', arial_reg))
pdfmetrics.registerFont(TTFont('Arial-Bold', arial_bold))
pdfmetrics.registerFont(TTFont('Arial-Italic', arial_italic))
pdfmetrics.registerFont(TTFont('Arial-BoldItalic', arial_bolditalic))

img_dir = "./test_imgs/"
reports_dir = "Reports"
os.makedirs(reports_dir, exist_ok=True) # creates output folder if it does not exist

In [388]:
print(reportlab.Version)

3.5.67


## Import the data to be reported from a csv file

This will be changed to a postgreSQL pull once the database is set up.

In [389]:
df = pd.read_csv('./test_input_tables/testIDs.csv')
print(df.head())

       HDOAReference        Host      Origin  morphospeciescode       FinalID  \
0  Interception 1103  Strawberry  California  HDOA260619_113_M3  Lorem Ipsum1   
1   Interception 111        Mint      Mexico  HDOA250917_034_M2  Lorem Ipsum2   
2   Interception 111        Mint      Mexico  HDOA250917_034_M1  Lorem Ipsum3   

   genusorlower FinalIDcommonname             DistributionNotes  HostNotes  \
0         False               Bug                    New for HI        NaN   
1          True              Bug2  Cosmopolitan, all HI islands        NaN   
2         False              Bug3  Cosmopolitan, all HI islands        NaN   

   speciescount  
0             4  
1             3  
2             6  


## Create a ReportLab Canvas with logos and basic text

In [390]:
# Create canvas with morphospeciescode filename in subfolder Reports
morphospeciescode = df['morphospeciescode'].head(1).iloc[0]
filename = str(morphospeciescode) + ".pdf"
full_path = os.path.join(reports_dir, filename)
c = canvas.Canvas(full_path, pagesize=letter) # Letter size; width: 612, height: 792 points
page_width, page_height = letter

UHIMlogo = "./logos/header1.png"
img_width = 530
img_height = 90
x_coord = (page_width - img_width) / 2 # to center image on page
c.drawImage(UHIMlogo, x_coord, 680, width=img_width, height=img_height, preserveAspectRatio=True)

x_center = page_width / 2 # to center text on page
c.setFont("Arial-Bold", 14) # font and size
c.drawCentredString(x_center, 655, "University of Hawaiʻi Insect Museum")
c.drawCentredString(x_center, 635, "Arthropod Identification Report")

c.setFont("Arial", 10)
today = date.today()
c.drawCentredString(x_center, 620, ("Report generated: " + today.strftime("%B %d, %Y")))

## Add information from csv file

In [391]:
c.setFont("Arial", 11)
# keep 12 y value spacing between text with font size 11

c.drawString(48, 586, "Project: Hawaiʻi Department of Agriculture and Biosecurity RFP-25-06-PI. Fiscal year 2025.")

HDOAreference = df['HDOAReference'].head(1).iloc[0]
c.drawString(48, 567, "HDAB reference: " + str(HDOAreference))
c.drawString(48, 555, "Date collected: 1/1/2026")
Origin = df['Origin'].head(1).iloc[0]
c.drawString(48, 543, "Origin: " + Origin)
Host = df['Host'].head(1).iloc[0]
c.drawString(48, 531, "Intercepted host: " + Host)
countspeciesinsample = df['speciescount'].head(1).iloc[0]
c.drawString(48, 519, "Number of species in sample: " + str(countspeciesinsample))

morphospeciescode = df['morphospeciescode'].head(1).iloc[0]
c.drawString(48, 495, "UHIM identification reference: " + str(morphospeciescode))
c.drawString(48, 483, "Number of specimens of this species: 12")

c.drawString(48, 459, "Integrative identification:")
genusorlower = df['genusorlower'].head(1).iloc[0]
if genusorlower == 1:
    c.setFont("Arial-Italic", 11)
else: c.setFont("Arial", 11)
c.drawString(169, 459, "Delia platura")
c.setFont("Arial", 11)
c.drawString(48, 447, "Common name: seedcorn maggot")
c.drawString(48, 435, "Order: Hemiptera")
c.drawString(48, 423, "Family: Miridae")

c.drawString(48, 399, "Identification notes: Camiel thinks this might be a spider")
c.drawString(48, 387, "Distribution notes: Only known from The Netherlands")

## Add photo of morphotype

In [392]:
#get a list of photos in the img_dir
jpg_filenames = [
    f for f in os.listdir(img_dir) 
    if f.lower().endswith('.jpg') and os.path.isfile(os.path.join(img_dir, f))
]
# Select image file names containing "morphospeciescode"
morphospeciesimages = [item for item in jpg_filenames if morphospeciescode in item]

if len(morphospeciesimages) == 0:
    print("No images found, continuing without image.")
else:
    # Select the first photo in the list to print
    speciesimage = os.path.join(img_dir, morphospeciesimages[0])
    # Print on page
    img_width = 500
    img_height = 400
    x_coord = (page_width - img_width) / 2
    c.drawImage(speciesimage, x_coord, 10, width=img_width, height=img_height, preserveAspectRatio=True)

    # Add image filename underneath
    imgfilename = os.path.basename(speciesimage)
    c.setFont("Arial", 10) # font and size
    c.drawCentredString(x_center, 60, imgfilename)

In [393]:
c.showPage() # finish page
c.save() # construct and save file to .pdf

# coordinate system:
#   y
#   |
#   |   page
#   | 
#   0-------x
